
# Model 2: Revenue Prediction

#Importing the required pacakages

In [0]:
%restart_python 


In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import joblib

#Importing the data

In [0]:
VOLUME_PATH = "/Volumes/workspace/default/raw_data/"

In [0]:
demographics = pd.read_csv(VOLUME_PATH + "customer_demographics.csv")
location = pd.read_csv(VOLUME_PATH + "customer_location.csv")
services = pd.read_csv(VOLUME_PATH + "customer_services.csv")
account_status = pd.read_csv(VOLUME_PATH + "customer_account_status.csv")
zipcode_population = pd.read_csv(VOLUME_PATH + "zipcode_population.csv")

#Cleaning

In [0]:
services.head(3)

,Customer ID,Offer,Phone Service,Avg Monthly Long Distance Charges,Multiple Lines,Internet Service,Internet Type,Avg Monthly GB Download,Online Security,Online Backup,Device Protection Plan,Premium Tech Support,Streaming TV,Streaming Movies,Streaming Music,Unlimited Data
0,0002-ORFBO,NaN,Yes,42.39,No,Yes,Cable,16.0,No,Yes,No,Yes,Yes,No,No,Yes
1,0003-MKNFE,NaN,Yes,10.69,Yes,Yes,Cable,10.0,No,No,No,No,No,Yes,Yes,No
2,0004-TLHLJ,Offer E,Yes,33.65,No,Yes,Fiber Optic,30.0,No,No,Yes,No,No,No,No,Yes


In [0]:
services.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 16 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Offer                              3166 non-null   object 
 2   Phone Service                      7043 non-null   object 
 3   Avg Monthly Long Distance Charges  6361 non-null   float64
 4   Multiple Lines                     6361 non-null   object 
 5   Internet Service                   7043 non-null   object 
 6   Internet Type                      5517 non-null   object 
 7   Avg Monthly GB Download            5517 non-null   float64
 8   Online Security                    5517 non-null   object 
 9   Online Backup                      5517 non-null   object 
 10  Device Protection Plan             5517 non-null   object 
 11  Premium Tech Support               5517 non-null   objec

In [0]:
services['Internet_Type_Clean'] = services['Internet Type'].fillna('No Internet Service')
services['Offer_Clean'] = services['Offer'].fillna('No Offer')


In [0]:
def missing_value_imp(x):
  if x.dtype == 'int' or x.dtype == 'float':
    x = x.fillna(x.mean())
  else:
    x = x.fillna(x.mode()[0])
  return x

In [0]:
services = services.apply(missing_value_imp)

In [0]:
services.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 18 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Offer                              7043 non-null   object 
 2   Phone Service                      7043 non-null   object 
 3   Avg Monthly Long Distance Charges  7043 non-null   float64
 4   Multiple Lines                     7043 non-null   object 
 5   Internet Service                   7043 non-null   object 
 6   Internet Type                      7043 non-null   object 
 7   Avg Monthly GB Download            7043 non-null   float64
 8   Online Security                    7043 non-null   object 
 9   Online Backup                      7043 non-null   object 
 10  Device Protection Plan             7043 non-null   object 
 11  Premium Tech Support               7043 non-null   objec

In [0]:
account_status.head(3)

,Customer ID,Number of Referrals,Tenure in Months,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,2,9,One Year,Yes,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,0,9,Month-to-Month,No,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,0,4,Month-to-Month,Yes,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices


In [0]:
account_status.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Customer ID                  7043 non-null   object 
 1   Number of Referrals          7043 non-null   int64  
 2   Tenure in Months             7043 non-null   int64  
 3   Contract                     7043 non-null   object 
 4   Paperless Billing            7043 non-null   object 
 5   Payment Method               7043 non-null   object 
 6   Monthly Charge               7043 non-null   float64
 7   Total Charges                7043 non-null   float64
 8   Total Refunds                7043 non-null   float64
 9   Total Extra Data Charges     7043 non-null   int64  
 10  Total Long Distance Charges  7043 non-null   float64
 11  Total Revenue                7043 non-null   float64
 12  Customer Status              7043 non-null   object 
 13  Churn Category    

In [0]:
account_status['Churn_Category_Clean'] = account_status['Churn Category'].fillna('Not Churned')
account_status['Churn_Reason_Clean'] = account_status['Churn Reason'].fillna('Not Churned')

In [0]:
account_status.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Customer ID                  7043 non-null   object 
 1   Number of Referrals          7043 non-null   int64  
 2   Tenure in Months             7043 non-null   int64  
 3   Contract                     7043 non-null   object 
 4   Paperless Billing            7043 non-null   object 
 5   Payment Method               7043 non-null   object 
 6   Monthly Charge               7043 non-null   float64
 7   Total Charges                7043 non-null   float64
 8   Total Refunds                7043 non-null   float64
 9   Total Extra Data Charges     7043 non-null   int64  
 10  Total Long Distance Charges  7043 non-null   float64
 11  Total Revenue                7043 non-null   float64
 12  Customer Status              7043 non-null   object 
 13  Churn Category    

In [0]:
account_status['Has_Discount'] = np.where(account_status['Monthly Charge'] < 0, 1, 0)
account_status['Monthly_Discount_Amount'] = np.where(
    account_status['Monthly Charge'] < 0, account_status['Monthly Charge'].abs(), 0)

#Merge into one master table

In [0]:
location.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Customer ID  7043 non-null   object 
 1   City         7043 non-null   object 
 2   Zip Code     7043 non-null   int64  
 3   Latitude     7043 non-null   float64
 4   Longitude    7043 non-null   float64
dtypes: float64(2), int64(1), object(2)
memory usage: 275.2+ KB


In [0]:
data = demographics.merge(location, on='Customer ID', how='left')
data = data.merge(services, on='Customer ID', how='left')
data = data.merge(account_status, on='Customer ID', how='left')
data = data.merge(zipcode_population, on='Zip Code', how='left')

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 45 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Offer                              7043 non-null   object 
 10  Phone Service                      7043 non-null   object 
 11  Avg Monthly Long Distance Charges  7043 non-null   float

In [0]:
data.columns

Index(['Customer ID', 'Gender', 'Age', 'Married', 'Number of Dependents',
       'City', 'Zip Code', 'Latitude', 'Longitude', 'Offer', 'Phone Service',
       'Avg Monthly Long Distance Charges', 'Multiple Lines',
       'Internet Service', 'Internet Type', 'Avg Monthly GB Download',
       'Online Security', 'Online Backup', 'Device Protection Plan',
       'Premium Tech Support', 'Streaming TV', 'Streaming Movies',
       'Streaming Music', 'Unlimited Data', 'Internet_Type_Clean',
       'Offer_Clean', 'Number of Referrals', 'Tenure in Months', 'Contract',
       'Paperless Billing', 'Payment Method', 'Monthly Charge',
       'Total Charges', 'Total Refunds', 'Total Extra Data Charges',
       'Total Long Distance Charges', 'Total Revenue', 'Customer Status',
       'Churn Category', 'Churn Reason', 'Churn_Category_Clean',
       'Churn_Reason_Clean', 'Has_Discount', 'Monthly_Discount_Amount',
       'Population'],
      dtype='object')

In [0]:
data.drop(columns=['Offer','Internet Type','Churn Category','Churn Reason'], inplace=True)

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 41 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Phone Service                      7043 non-null   object 
 10  Avg Monthly Long Distance Charges  7043 non-null   float64
 11  Multiple Lines                     7043 non-null   objec

In [0]:
data.columns = data.columns.str.replace(' ','_')

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 41 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer_ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number_of_Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip_Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Phone_Service                      7043 non-null   object 
 10  Avg_Monthly_Long_Distance_Charges  7043 non-null   float64
 11  Multiple_Lines                     7043 non-null   objec

In [0]:
data.head(3)

,Customer_ID,Gender,Age,Married,Number_of_Dependents,City,Zip_Code,Latitude,Longitude,Phone_Service,Avg_Monthly_Long_Distance_Charges,Multiple_Lines,Internet_Service,Avg_Monthly_GB_Download,Online_Security,Online_Backup,Device_Protection_Plan,Premium_Tech_Support,Streaming_TV,Streaming_Movies,Streaming_Music,Unlimited_Data,Internet_Type_Clean,Offer_Clean,Number_of_Referrals,Tenure_in_Months,Contract,Paperless_Billing,Payment_Method,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue,Customer_Status,Churn_Category_Clean,Churn_Reason_Clean,Has_Discount,Monthly_Discount_Amount,Population
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,Yes,42.39,No,Yes,16.0,No,Yes,No,Yes,Yes,No,No,Yes,Cable,No Offer,2,9,One Year,Yes,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,Not Churned,Not Churned,0,0.0,4498
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,Yes,10.69,Yes,Yes,10.0,No,No,No,No,No,Yes,Yes,No,Cable,No Offer,0,9,Month-to-Month,No,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,Not Churned,Not Churned,1,4.0,31297
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,Yes,33.65,No,Yes,30.0,No,No,Yes,No,No,No,No,Yes,Fiber Optic,Offer E,0,4,Month-to-Month,Yes,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices,0,0.0,62069


In [0]:
data['Gender'] = pd.get_dummies(data['Gender'], drop_first=True, dtype='int')
data['Married'] = np.where(data['Married'] == 'Yes', 1, 0)
data['Phone_Service'] = np.where(data['Phone_Service'] == 'Yes', 1, 0)
data['Multiple_Lines'] = np.where(data['Multiple_Lines'] == 'Yes', 1, 0)
data['Internet_Service'] = np.where(data['Internet_Service'] == 'Yes', 1, 0)
data['Online_Security'] = np.where(data['Online_Security'] == 'Yes', 1, 0)
data['Online_Backup'] = np.where(data['Online_Backup'] == 'Yes', 1, 0)
data['Device_Protection_Plan'] = np.where(data['Device_Protection_Plan'] == 'Yes', 1, 0)
data['Premium_Tech_Support'] = np.where(data['Premium_Tech_Support'] == 'Yes', 1, 0)
data['Streaming_TV'] = np.where(data['Streaming_TV'] == 'Yes', 1, 0)
data['Streaming_Movies'] = np.where(data['Streaming_Movies'] == 'Yes', 1, 0)
data['Streaming_Music'] = np.where(data['Streaming_Music'] == 'Yes', 1, 0)
data['Unlimited_Data'] = np.where(data['Unlimited_Data'] == 'Yes', 1, 0)
data = pd.concat([data, pd.get_dummies(data['Internet_Type_Clean'], drop_first=True,dtype='int', prefix='Internet_Type_Clean')], axis=1)
data.drop('Internet_Type_Clean', axis=1, inplace=True)
data = pd.concat([data, pd.get_dummies(data['Offer_Clean'], drop_first=True, dtype='int',prefix='Offer_Clean')], axis=1)
data.drop('Offer_Clean', axis=1, inplace=True)
data = pd.concat([data, pd.get_dummies(data['Contract'], drop_first=True,dtype='int', prefix='Contract')], axis=1)
data.drop('Contract', axis=1, inplace=True)
data['Paperless_Billing'] = np.where(data['Paperless_Billing'] == 'Yes', 1, 0)
data = pd.concat([data, pd.get_dummies(data['Payment_Method'], drop_first=True,dtype='int', prefix='Payment_Method')], axis=1)
data.drop('Payment_Method', axis=1, inplace=True)

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 49 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Customer_ID                              7043 non-null   object 
 1   Gender                                   7043 non-null   int64  
 2   Age                                      7043 non-null   int64  
 3   Married                                  7043 non-null   int64  
 4   Number_of_Dependents                     7043 non-null   int64  
 5   City                                     7043 non-null   object 
 6   Zip_Code                                 7043 non-null   int64  
 7   Latitude                                 7043 non-null   float64
 8   Longitude                                7043 non-null   float64
 9   Phone_Service                            7043 non-null   int64  
 10  Avg_Monthly_Long_Distance_Charges        7043 no

In [0]:
data.columns = data.columns.str.replace(' ','_')


In [0]:
X = data.drop(['Customer_ID','City','Total_Revenue','Churn_Category_Clean','Churn_Reason_Clean','Customer_Status'], axis=1)
y = data['Total_Revenue']

# Spliting the data

In [0]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

# Feature engineering

In [0]:
vif = pd.DataFrame()
vif['feature'] = x_train.columns
vif['score'] = [ variance_inflation_factor(x_train.values, i) for i in range(len(x_train.columns))]

/local_disk0/.ephemeral_nfs/envs/pythonEnv-a39a5df5-1257-474e-b72f-affebf44ade1/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a39a5df5-1257-474e-b72f-affebf44ade1/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a39a5df5-1257-474e-b72f-affebf44ade1/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a39a5df5-1257-474e-b72f-affebf44ade1/lib/p

In [0]:
vif.loc[vif['score']<5, 'feature'].values

array(['Gender', 'Age', 'Married', 'Number_of_Dependents',
       'Phone_Service', 'Avg_Monthly_Long_Distance_Charges',
       'Multiple_Lines', 'Avg_Monthly_GB_Download', 'Online_Security',
       'Online_Backup', 'Device_Protection_Plan', 'Premium_Tech_Support',
       'Streaming_TV', 'Streaming_Music', 'Unlimited_Data',
       'Number_of_Referrals', 'Paperless_Billing', 'Total_Refunds',
       'Total_Extra_Data_Charges', 'Monthly_Discount_Amount',
       'Population', 'Internet_Type_Clean_DSL', 'Offer_Clean_Offer_A',
       'Offer_Clean_Offer_B', 'Offer_Clean_Offer_C',
       'Offer_Clean_Offer_D', 'Offer_Clean_Offer_E', 'Contract_One_Year',
       'Contract_Two_Year', 'Payment_Method_Credit_Card',
       'Payment_Method_Mailed_Check'], dtype=object)

In [0]:
x = data[['Gender', 'Age', 'Married', 'Number_of_Dependents',
       'Phone_Service', 'Avg_Monthly_Long_Distance_Charges',
       'Multiple_Lines', 'Avg_Monthly_GB_Download', 'Online_Security',
       'Online_Backup', 'Device_Protection_Plan', 'Premium_Tech_Support',
       'Streaming_TV', 'Streaming_Music', 'Unlimited_Data',
       'Number_of_Referrals', 'Paperless_Billing', 'Total_Refunds',
       'Total_Extra_Data_Charges', 'Monthly_Discount_Amount',
       'Population', 'Internet_Type_Clean_DSL', 'Offer_Clean_Offer_A',
       'Offer_Clean_Offer_B', 'Offer_Clean_Offer_C',
       'Offer_Clean_Offer_D', 'Offer_Clean_Offer_E', 'Contract_One_Year',
       'Contract_Two_Year', 'Payment_Method_Credit_Card',
       'Payment_Method_Mailed_Check']]

# Spliting the data by feature scaling

In [0]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=123)

# Model building

# Linear regression

In [0]:
param_grid_lr = {
    'fit_intercept': [True, False],
    'positive': [True, False]
}

In [0]:
grid_lr = GridSearchCV(LinearRegression(),param_grid_lr,cv=5,scoring='r2')
grid_lr.fit(x_train, y_train)

GridSearchCV(cv=5, estimator=LinearRegression(),
             param_grid={'fit_intercept': [True, False],
                         'positive': [True, False]},
             scoring='r2')

In [0]:
lin = grid_lr.best_estimator_
lin.fit(x_train,y_train)

LinearRegression()

In [0]:
preds = lin.predict(x_test)
print(f'Linear Regression -- R2: {r2_score(y_test, preds):.3f}, MAE: {mean_absolute_error(y_test, preds):.2f}')

Linear Regression -- R2: 0.752, MAE: 1136.16


# Random forest

In [0]:
par_grid_rfr = {'n_estimators': [100], 'max_depth': [6, 10]}

In [0]:
grid_rfr = GridSearchCV(RandomForestRegressor(n_jobs=-1, random_state=123), param_grid=par_grid_rfr, cv=3, scoring='r2', n_jobs=-1)

In [0]:
grid_rfr = grid_rfr.fit(x_train, y_train)

In [0]:
rfr = grid_rfr.best_estimator_
rfr.fit(x_train, y_train)


RandomForestRegressor(max_depth=10, n_jobs=-1, random_state=123)

In [0]:
preds = rfr.predict(x_test)
print(f'Random Forest -- R2: {r2_score(y_test, preds):.3f}, MAE: {mean_absolute_error(y_test, preds):.2f}')

Random Forest -- R2: 0.791, MAE: 952.74


# XGBoost

In [0]:
par_grid_xgbr = {'n_estimators': [100], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}

In [0]:
grid_xgbr = GridSearchCV(XGBRegressor(random_state=123), param_grid=par_grid_xgbr, cv=3, scoring='r2', n_jobs=-1)

In [0]:
grid_xgbr = grid_xgbr.fit(x_train, y_train)

In [0]:
xgbr = grid_xgbr.best_estimator_
xgbr.fit(x_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [0]:
preds = xgbr.predict(x_test)
print(f'XGBoost -- R2: {r2_score(y_test, preds):.3f}, MAE: {mean_absolute_error(y_test, preds):.2f}')

XGBoost -- R2: 0.826, MAE: 867.59


# LightGBM

In [0]:
par_grid_lgbr = {'n_estimators': [100], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}


In [0]:
grid_lgbr = GridSearchCV(LGBMRegressor(random_state=123, verbose=-1), param_grid=par_grid_lgbr, cv=3, scoring='r2', n_jobs=-1)

In [0]:
grid_lgbr = grid_lgbr.fit(x_train, y_train)

In [0]:
lgbmr = grid_lgbr.best_estimator_
lgbmr.fit(x_train, y_train)

LGBMRegressor(max_depth=5, random_state=123, verbose=-1)

In [0]:
preds = lgbmr.predict(x_test)
print(f'LightGBM -- R2: {r2_score(y_test, preds):.3f}, MAE: {mean_absolute_error(y_test, preds):.2f}')

LightGBM -- R2: 0.823, MAE: 874.17


# Comparing all the model prediction

In [0]:
score = pd.DataFrame([grid_lr.best_score_, grid_rfr.best_score_, grid_xgbr.best_score_, grid_lgbr.best_score_])
name = pd.DataFrame(['linear_regression', 'random_forest', 'xgboost', 'lightgbm'])
best_score = pd.concat([name, score], axis=1)
best_score.columns = ['model', 'r2']
best_score.sort_values('r2', ascending=False)

,model,r2
2,xgboost,0.812303
3,lightgbm,0.806917
1,random_forest,0.779202
0,linear_regression,0.747629


# Saving the best model to pkl file

In [0]:
joblib.dump(xgbr, '/Volumes/workspace/default/raw_data/revenue_model.pkl')

['/Volumes/workspace/default/raw_data/revenue_model.pkl']

In [0]:
joblib.load('/Volumes/workspace/default/raw_data/revenue_model.pkl')

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

#End